# Đề xuất đặc trưng và bảng nhóm biến Ames Housing

**Người phụ trách:** Dương Hoàng Anh — MSSV 0364169.  
**Mốc công việc:** 21–23/11/2026.

Bàn giao bảng **79 biến gốc thuộc 12 nhóm**, **9 đặc trưng ưu tiên** và **12 đặc trưng thử thêm**.
Notebook kiểm tra công thức trên tập train đã chia sẵn, không huấn luyện hoặc đánh giá mô hình.

Tài liệu giải thích đầy đủ nằm ở `01_Report/HoangAnh/De_xuat_dac_trung_va_nhom_bien.md`.
Khi ghép mô hình, Hinh đưa các quy tắc đã chọn vào custom transformer trong `src/` trước `ColumnTransformer`.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

source_root = Path.cwd()
if source_root.name == "notebooks":
    source_root = source_root.parent
if (source_root / "03_SourceCode" / "src").is_dir():
    source_root = source_root / "03_SourceCode"
assert (source_root / "src" / "feature_proposal.py").is_file()
if str(source_root) not in sys.path:
    sys.path.insert(0, str(source_root))

from src.feature_proposal import (
    GROUPS,
    PROPOSALS,
    feature_group_table,
    feature_proposal_table,
    feature_summary,
    preview_features,
    read_training_features,
)

pd.set_option("display.max_columns", 12)
print("Python:", sys.version.split()[0])


Python: 3.11.9


## 1 Tập dữ liệu dùng để đề xuất

Loại các Id có trong `data/processed/test_ids.csv`, rồi bỏ `Id` và `SalePrice`.
Mọi bảng thống kê dưới đây chỉ dùng 1.168 căn train trước khi lọc ngoại lai.
Các công thức tính riêng từng dòng, không học median, không scale và không encode nhãn.

In [2]:
X_train = read_training_features()
original = X_train.copy(deep=True)
groups = feature_group_table(X_train)
proposals = feature_proposal_table()
preview = preview_features(X_train)

assert X_train.shape[1] == 79
assert not {"Id", "SalePrice"}.intersection(X_train.columns)
assert groups["bien"].is_unique and len(groups) == 79
assert set(groups["bien"]) == set(X_train.columns)
assert preview.index.equals(X_train.index)
print("Tập train:", X_train.shape)
print(
    "Kiểu lưu trữ:",
    X_train.select_dtypes(include="number").shape[1],
    "cột số và",
    X_train.select_dtypes(exclude="number").shape[1],
    "cột nhãn",
)
print("Số nhóm:", len(GROUPS), "| Đặc trưng đề xuất:", len(PROPOSALS))
print("Phân tích không sử dụng nhãn giá bán hoặc điểm đánh giá holdout.")


Tập train: (1168, 79)
Kiểu lưu trữ: 36 cột số và 43 cột nhãn
Số nhóm: 12 | Đặc trưng đề xuất: 21
Phân tích không sử dụng nhãn giá bán hoặc điểm đánh giá holdout.


## 2 Bảng nhóm biến

Mỗi biến xuất hiện trong một nhóm chính; công thức FE có thể dùng nhiều nhóm.
`MSSubClass` là mã phân loại; `MoSold` là tháng có chu kỳ. Baseline one-hot hai biến này
cùng các nhãn còn lại, tương ứng 34 biến số và 45 biến phân loại trước one-hot.
Các cột chất lượng có thể thử ordinal sau khi nhóm thống nhất thứ tự.

In [3]:
group_summary = pd.DataFrame(
    [
        {"nhom": name, "so_bien": len(columns), "cac_bien": ", ".join(columns)}
        for name, columns in GROUPS.items()
    ]
)
assert group_summary["so_bien"].sum() == 79
with pd.option_context("display.max_colwidth", 180):
    display(group_summary)


,nhom,so_bien,cac_bien
0,Vị trí và lô đất,12,"MSZoning, LotFrontage, LotArea, Street, Alley, LotShape, LandContour, LotConfig, LandSlope, Neighborhood, Condition1, Condition2"
1,Loại hình nhà,3,"MSSubClass, BldgType, HouseStyle"
2,Chất lượng và kết cấu,11,"OverallQual, OverallCond, RoofStyle, RoofMatl, Exterior1st, Exterior2nd, MasVnrType, MasVnrArea, ExterQual, ExterCond, Foundation"
3,Thời gian,5,"YearBuilt, YearRemodAdd, GarageYrBlt, MoSold, YrSold"
4,Diện tích và phòng trên mặt đất,8,"1stFlrSF, 2ndFlrSF, LowQualFinSF, GrLivArea, BedroomAbvGr, KitchenAbvGr, KitchenQual, TotRmsAbvGrd"
5,Tầng hầm,11,"BsmtQual, BsmtCond, BsmtExposure, BsmtFinType1, BsmtFinSF1, BsmtFinType2, BsmtFinSF2, BsmtUnfSF, TotalBsmtSF, BsmtFullBath, BsmtHalfBath"
6,Phòng tắm trên mặt đất,2,"FullBath, HalfBath"
7,Hệ thống và tiện nghi trong nhà,8,"Utilities, Heating, HeatingQC, CentralAir, Electrical, Functional, Fireplaces, FireplaceQu"
8,Garage và lối xe,7,"GarageType, GarageFinish, GarageCars, GarageArea, GarageQual, GarageCond, PavedDrive"
9,Không gian ngoài trời,8,"WoodDeckSF, OpenPorchSF, EnclosedPorch, 3SsnPorch, ScreenPorch, PoolArea, PoolQC, Fence"


In [4]:
# Bảng đầy đủ có ý nghĩa, cách mã hóa, xử lý thiếu và số NA trên train.
with pd.option_context("display.max_rows", 100, "display.max_colwidth", 85):
    display(groups)


,bien,nhom,y_nghia,kieu_trong_csv,kieu_ngu_nghia,tien_xu_ly_de_xuat,xu_ly_thieu,so_thieu_train,ty_le_thieu_train_pct
0,MSSubClass,Loại hình nhà,"Mã loại nhà; số là nhãn, không phải đại lượng liên tục",Số,Mã phân loại,"Đổi sang chuỗi, one-hot; không scale mã số","Thiếu thành Unknown sau đổi chuỗi; hoặc mode trong fold, không median mã",0,0.00
1,MSZoning,Vị trí và lô đất,Phân vùng sử dụng đất,Nhãn,Phân loại,"One-hot, handle_unknown='ignore'; fit trong fold",Unknown hoặc most_frequent học trong fold; không xóa dòng,0,0.00
2,LotFrontage,Vị trí và lô đất,"Chiều dài mặt tiền tiếp giáp đường, feet",Số,Đại lượng hoặc số đếm,Median trong fold; scale cho Ridge/KNN,Median trong fold; median theo Neighborhood cũng phải fit trong fold,217,18.58
3,LotArea,Vị trí và lô đất,"Diện tích lô đất, ft²",Số,Đại lượng hoặc số đếm,Median trong fold; scale cho Ridge/KNN,"Giữ 0 đã ghi; thiếu thật dùng median trong fold, không mặc định 0",0,0.00
4,Street,Vị trí và lô đất,Loại mặt đường,Nhãn,Phân loại,"One-hot, handle_unknown='ignore'; fit trong fold",Unknown hoặc most_frequent học trong fold; không xóa dòng,0,0.00
5,Alley,Vị trí và lô đất,Loại đường ngõ; NA theo mô tả là không có ngõ,Nhãn,Phân loại,"One-hot, handle_unknown='ignore'; fit trong fold",NA thành None theo mô tả dữ liệu; giữ trạng thái không có,1094,93.66
6,LotShape,Vị trí và lô đất,Hình dạng lô đất,Nhãn,Phân loại,"One-hot, handle_unknown='ignore'; fit trong fold",Unknown hoặc most_frequent học trong fold; không xóa dòng,0,0.00
7,LandContour,Vị trí và lô đất,Địa hình lô đất,Nhãn,Phân loại,"One-hot, handle_unknown='ignore'; fit trong fold",Unknown hoặc most_frequent học trong fold; không xóa dòng,0,0.00
8,Utilities,Hệ thống và tiện nghi trong nhà,Các dịch vụ hạ tầng được cung cấp,Nhãn,Phân loại,"One-hot, handle_unknown='ignore'; fit trong fold",Unknown hoặc most_frequent học trong fold; không xóa dòng,0,0.00
9,LotConfig,Vị trí và lô đất,Vị trí và cách bố trí lô đất,Nhãn,Phân loại,"One-hot, handle_unknown='ignore'; fit trong fold",Unknown hoặc most_frequent học trong fold; không xóa dòng,0,0.00


## 3 Công thức đặc trưng đề xuất

`TotalSF` cộng tầng hầm và hai tầng chính, có cả hầm chưa hoàn thiện, không cộng thêm `GrLivArea`
và không cộng riêng `LowQualFinSF`. `TotalBath` dùng trọng số 0,5 cho half-bath như một quy ước.
Tuổi tính theo năm bán từng dòng. Không đủ dữ liệu thì giữ NA để Pipeline impute trong từng fold.
Các giả thuyết cần được kiểm chứng bằng CV, không kết luận tăng độ chính xác từ thống kê mô tả.

In [5]:
core = proposals.loc[proposals["uu_tien"] == "Cốt lõi"]
optional = proposals.loc[proposals["uu_tien"] == "Thử thêm"]
assert len(core) == 9 and len(optional) == 12
with pd.option_context("display.max_colwidth", 130):
    display(core.reset_index(drop=True))
    display(optional.reset_index(drop=True))


,dac_trung,bien_nguon,cong_thuc,don_vi,uu_tien,y_nghia_va_gia_thuyet,xu_ly_thieu_va_ngoai_le
0,TotalSF,"TotalBsmtSF, 1stFlrSF, 2ndFlrSF",TotalBsmtSF + 1stFlrSF + 2ndFlrSF,ft²,Cốt lõi,Tổng quy mô tầng hầm và hai tầng chính; gồm hầm chưa hoàn thiện,Thiếu một thành phần thì NA; không cộng thêm GrLivArea
1,HouseAge,"YrSold, YearBuilt",YrSold - YearBuilt,năm,Cốt lõi,"Tuổi nhà tại năm bán, giả thuyết phản ánh hao mòn","Thiếu năm, năm không hợp lệ hoặc tuổi âm thì NA"
2,RemodelAge,"YrSold, YearRemodAdd",YrSold - YearRemodAdd,năm,Cốt lõi,Thời gian từ lần cải tạo đến năm bán,Tuổi âm thì NA; giữ nguyên năm gốc để nhóm đối chiếu
3,TotalBath,"FullBath, HalfBath, BsmtFullBath, BsmtHalfBath",FullBath + BsmtFullBath + 0.5 * (HalfBath + BsmtHalfBath),phòng quy đổi,Cốt lõi,Gộp tiện nghi tắm ở cả trên và dưới đất,Thiếu thành phần thì NA; trọng số 0.5 là quy ước đặc trưng
4,TotalOutdoorSF,"WoodDeckSF, OpenPorchSF, EnclosedPorch, 3SsnPorch, ScreenPorch",WoodDeckSF + OpenPorchSF + EnclosedPorch + 3SsnPorch + ScreenPorch,ft²,Cốt lõi,"Quy mô sàn gỗ và các loại hiên, không gồm hồ bơi",Thiếu thành phần thì NA; số 0 có sẵn được giữ nguyên
5,HasGarage,"GarageCars, GarageArea",1 nếu GarageCars > 0 hoặc GarageArea > 0; 0 nếu cả hai = 0,0/1,Cốt lõi,Tách thông tin có garage khỏi quy mô garage,Không đủ bằng chứng có hoặc không có thì NA
6,HasBsmt,TotalBsmtSF,1 nếu TotalBsmtSF > 0; 0 nếu = 0,0/1,Cốt lõi,Phân biệt không có hầm với hầm có diện tích khác nhau,"Diện tích thiếu hoặc âm thì NA, không tự gán 0"
7,HasFireplace,Fireplaces,1 nếu Fireplaces > 0; 0 nếu = 0,0/1,Cốt lõi,Biểu diễn tiện nghi lò sưởi theo trạng thái có hay không,Số lượng thiếu hoặc âm thì NA
8,HasRemodel,"YearBuilt, YearRemodAdd, YrSold",1 nếu YearRemodAdd > YearBuilt; 0 nếu bằng nhau,0/1,Cốt lõi,Phân biệt nhà đã cải tạo với nhà chưa cải tạo,Thiếu năm hoặc không thỏa YearBuilt <= YearRemodAdd <= YrSold thì NA


,dac_trung,bien_nguon,cong_thuc,don_vi,uu_tien,y_nghia_va_gia_thuyet,xu_ly_thieu_va_ngoai_le
0,HasPool,PoolArea,1 nếu PoolArea > 0; 0 nếu = 0,0/1,Thử thêm,Tách tiện nghi hồ bơi; cần kiểm tra độ hiếm trong CV,Diện tích thiếu hoặc âm thì NA
1,Has2ndFloor,2ndFlrSF,1 nếu 2ndFlrSF > 0; 0 nếu = 0,0/1,Thử thêm,Phân biệt nhà có tầng hai,Diện tích thiếu hoặc âm thì NA
2,GarageAge,"YrSold, GarageYrBlt, GarageCars, GarageArea",YrSold - GarageYrBlt khi HasGarage = 1,năm,Thử thêm,"Tuổi garage khi bán, có thể khác tuổi nhà","Không garage, thiếu năm hoặc tuổi âm thì NA; giữ HasGarage đi kèm"
3,FinishedBsmtSF,"BsmtFinSF1, BsmtFinSF2",BsmtFinSF1 + BsmtFinSF2,ft²,Thử thêm,Quy mô phần tầng hầm đã hoàn thiện,Thiếu thành phần thì NA; không có hầm và số liệu đều 0 thì giữ 0
4,FinishedBsmtRatio,"BsmtFinSF1, BsmtFinSF2, TotalBsmtSF",(BsmtFinSF1 + BsmtFinSF2) / TotalBsmtSF,tỷ lệ 0 đến 1,Thử thêm,"Tỷ lệ hoàn thiện tầng hầm, bổ sung thông tin ngoài tổng diện tích","Mẫu số <= 0, thiếu dữ liệu hoặc tỷ lệ > 1 thì NA; giữ HasBsmt"
5,LivingLotRatio,"GrLivArea, LotArea",GrLivArea / LotArea,tỷ số,Thử thêm,Diện tích ở trên mặt đất tương đối với diện tích đất,LotArea <= 0 hoặc thiếu thì NA; tỷ số > 1 vẫn có thể hợp lệ
6,GarageAreaPerCar,"GarageArea, GarageCars",GarageArea / GarageCars,ft²/xe,Thử thêm,Độ rộng garage trên mỗi chỗ xe,GarageCars <= 0 hoặc thiếu thì NA; giữ HasGarage đi kèm
7,QualArea,"OverallQual, GrLivArea",OverallQual * GrLivArea,điểm × ft²,Thử thêm,"Tương tác chất lượng và quy mô, hữu ích để thử với Ridge","Thiếu dữ liệu, chất lượng ngoài 1 đến 10 hoặc diện tích âm thì NA"
8,LogGrLivArea,GrLivArea,log1p(GrLivArea),log tự nhiên,Thử thêm,Thử quan hệ diện tích với giá ở thang log,Thiếu hoặc diện tích âm thì NA; không cần học tham số từ dữ liệu
9,LogLotArea,LotArea,log1p(LotArea),log tự nhiên,Thử thêm,Giảm tác động của đuôi dài diện tích đất,Thiếu hoặc LotArea <= 0 thì NA


## 4 Thống kê kiểm tra công thức trên train

Bảng dưới chưa impute. NA ở tuổi garage và tỷ số có thể biểu thị không có hạng mục,
cần giữ các biến `HasGarage` và `HasBsmt` đi kèm.
Căn Id 524 trong train có năm cải tạo 2008 sau năm bán 2007; `RemodelAge` và `HasRemodel` giữ NA.

In [6]:
summary = feature_summary(X_train)
display(summary.round(4))

bad_years = X_train["YearRemodAdd"] > X_train["YrSold"]
display(
    pd.concat(
        [
            X_train.loc[bad_years, ["YearBuilt", "YearRemodAdd", "YrSold"]],
            preview.loc[bad_years, ["HouseAge", "RemodelAge", "HasRemodel"]],
        ],
        axis=1,
    )
)


,dac_trung,so_dong_train,so_thieu,nho_nhat,trung_vi,lon_nhat
0,TotalSF,1168,0,334.0000,2494.0000,11752.0000
1,HouseAge,1168,0,0.0000,35.0000,136.0000
2,RemodelAge,1168,1,0.0000,14.0000,60.0000
3,TotalBath,1168,0,1.0000,2.0000,6.0000
4,TotalOutdoorSF,1168,0,0.0000,168.0000,1027.0000
5,HasGarage,1168,0,0.0000,1.0000,1.0000
6,HasBsmt,1168,0,0.0000,1.0000,1.0000
7,HasFireplace,1168,0,0.0000,1.0000,1.0000
8,HasRemodel,1168,1,0.0000,0.0000,1.0000
9,HasPool,1168,0,0.0000,0.0000,1.0000


,YearBuilt,YearRemodAdd,YrSold,HouseAge,RemodelAge,HasRemodel
523,2007,2008,2007,0.0,NaN,NaN


In [7]:
# Minh họa 5 căn train; không đưa Id hoặc SalePrice vào phép tính.
source_columns = [
    "TotalBsmtSF",
    "1stFlrSF",
    "2ndFlrSF",
    "YrSold",
    "YearBuilt",
    "FullBath",
    "HalfBath",
    "BsmtFullBath",
    "BsmtHalfBath",
]
display(
    pd.concat(
        [
            X_train[source_columns].head(),
            preview[["TotalSF", "HouseAge", "TotalBath"]].head(),
        ],
        axis=1,
    )
)


,TotalBsmtSF,1stFlrSF,2ndFlrSF,YrSold,YearBuilt,FullBath,HalfBath,BsmtFullBath,BsmtHalfBath,TotalSF,HouseAge,TotalBath
0,856,856,854,2008,2003,2,1,1,0,2566.0,5.0,3.5
1,1262,1262,0,2007,1976,2,0,0,1,2524.0,31.0,2.5
2,920,920,866,2008,2001,2,1,1,0,2706.0,7.0,3.5
3,756,961,756,2006,1915,1,0,1,0,2473.0,91.0,2.0
4,1145,1145,1053,2008,2000,2,1,1,0,3343.0,8.0,3.5


## 5 Kiểm tra tính hợp lệ và tình huống biên

Kiểm tra dữ liệu đầu vào không bị sửa, thứ tự dòng được giữ, không có vô cực,
cờ chỉ có giá trị 0/1/NA và phép tính không phụ thuộc các dòng khác.
Các tình huống nhân tạo dưới đây kiểm tra không có garage/hầm, thiếu thành phần cộng,
tuổi âm, trạng thái garage chưa biết và tháng không hợp lệ. Chúng không ghi vào dữ liệu gốc.

In [8]:
pd.testing.assert_frame_equal(original, X_train)
assert not np.isinf(preview.to_numpy()).any()
for column in [c for c in preview if c.startswith("Has")]:
    assert preview[column].dropna().isin([0, 1]).all()
for column in ["HouseAge", "RemodelAge", "GarageAge"]:
    assert preview[column].dropna().ge(0).all()
assert preview["FinishedBsmtRatio"].dropna().between(0, 1).all()
single = X_train.iloc[[0]]
pd.testing.assert_frame_equal(preview_features(single), preview.loc[single.index])
print("Đạt: dữ liệu gốc được giữ, các miền giá trị hợp lệ, tính độc lập theo dòng.")


Đạt: dữ liệu gốc được giữ, các miền giá trị hợp lệ, tính độc lập theo dòng.


In [9]:
cases = pd.concat([X_train.iloc[[0]]] * 5, ignore_index=True)
for column in cases.select_dtypes(include="number"):
    cases[column] = cases[column].astype(float)

cases.loc[
    0, ["GarageCars", "GarageArea", "TotalBsmtSF", "BsmtFinSF1", "BsmtFinSF2"]
] = 0
cases.loc[0, "GarageYrBlt"] = np.nan
cases.loc[1, "FullBath"] = np.nan
cases.loc[2, ["YearBuilt", "YearRemodAdd", "YrSold"]] = [2007, 2008, 2007]
cases.loc[3, "GarageCars"] = 0
cases.loc[3, "GarageArea"] = np.nan
cases.loc[4, "MoSold"] = 13
checked = preview_features(cases)

assert checked.loc[0, ["HasGarage", "HasBsmt"]].eq(0).all()
assert (
    checked.loc[0, ["GarageAge", "GarageAreaPerCar", "FinishedBsmtRatio"]].isna().all()
)
assert pd.isna(checked.loc[1, "TotalBath"])
assert checked.loc[2, "HouseAge"] == 0
assert checked.loc[2, ["RemodelAge", "HasRemodel"]].isna().all()
assert pd.isna(checked.loc[3, "HasGarage"])
assert checked.loc[4, ["MoSoldSin", "MoSoldCos"]].isna().all()
assert not np.isinf(checked.to_numpy()).any()
print("Đạt cả 5 tình huống biên; không thay dữ liệu gốc, không sinh vô cực.")


Đạt cả 5 tình huống biên; không thay dữ liệu gốc, không sinh vô cực.


## 6 Bàn giao cho Pipeline

1. So sánh biến gốc với biến gốc cộng 9 đặc trưng cốt lõi bằng cùng mô hình và các fold CV trên train.
2. Thử thêm từng nhóm trạng thái tiện nghi, tuổi/hoàn thiện, tỷ số, tương tác, log và tháng chu kỳ.
3. Tạo đặc trưng trước `ColumnTransformer`; chọn lại cột số/phân loại để các cột mới không bị bỏ qua.
4. Mọi imputer, scaler, encoder và bước chọn biến dựa vào dữ liệu phải fit trong từng fold train.
5. Giữ holdout cho lần đánh giá cuối. Khi đánh giá dự đoán học từ log giá, đổi về giá đúng thang đo.

Tham khảo [quy tắc chống rò rỉ dữ liệu của scikit-learn](https://scikit-learn.org/stable/common_pitfalls.html#data-leakage),
[ColumnTransformer](https://scikit-learn.org/stable/modules/compose.html#columntransformer-for-heterogeneous-data)
và `data/raw/data_description.txt` trong dự án.

Để tái tạo ba bảng CSV, từ `03_SourceCode` chạy:
```cmd
.venv\Scripts\python.exe -m src.feature_proposal
```